# Module 3 Lab 1: Rescue a Messy Dataset
**CS 82A — Santa Monica College** 
**Student:** Ramisha Tasfia 
**GitHub repo:** see Appendix

## Step 1: Load and size up the file

In [1]:
import pandas as pd

df = pd.read_csv('messy_sales.csv')
print('Shape:', df.shape)
display(df.head(10))
print('\nColumn types:')
print(df.dtypes)

Shape: (300, 6)


,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,NaN,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,NaN,4,30303
6,1204,04/18/2026,charger,8.41,2,2134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210



Column types:
order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object


## Step 2: First Observations

Three problems already visible in this data:

1. **Mixed date formats** — the `date` column contains two different formats side by side: `MM/DD/YYYY` (e.g., `04/06/2026`) and `YYYY-MM-DD` (e.g., `2026-05-30`). pandas reads this as plain text (`object` dtype), not real dates, which means sorting by date or doing any time-based analysis will fail.

2. **Missing prices** — row 3 (order 1066) and row 5 (order 1280) already show `NaN` in the `price` column in the first ten rows alone. Any revenue calculation run on this data as-is will silently undercount.

3. **Zip codes stored as integers, dropping leading zeros** — row 6 shows zip `2134` but the real zip code is `02134` (Boston area). Storing zip as `int64` strips the leading zero, making the data wrong for any geographic analysis or merging against a zip-code reference table.

## Step 3: Count the Missing Values

In [2]:
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

## Step 4: Impute the Missing Prices

In [3]:
# Use median, not mean — price distributions are often right-skewed
# (a few high-ticket items like monitors pull the mean up,
# so the median better represents a typical order price)
fill_value = df['price'].median()
print(f'Fill value used (median): {fill_value}')

df['price'] = df['price'].fillna(fill_value)
print('\nMissing values after imputation:')
print(df.isna().sum())

Fill value used (median): 37.53

Missing values after imputation:
order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64


**Checkpoint:** `df.isna().sum()` shows 0 everywhere. ✓

## Step 5: Remove Duplicate Rows

In [4]:
print('Duplicate rows found:', df.duplicated().sum())
before = df.shape
df = df.drop_duplicates()
print(f'Shape before drop_duplicates: {before}')
print(f'Shape after  drop_duplicates: {df.shape}')

Duplicate rows found: 8
Shape before drop_duplicates: (300, 6)
Shape after  drop_duplicates: (292, 6)


## Step 6: Repair the Zip Codes

In [5]:
df['zip'] = df['zip'].astype(str).str.zfill(5)
print('Sample zips after repair:')
df['zip'].head(10)

Sample zips after repair:


0    60614
1    30303
2    10001
3    98101
4    90405
5    30303
6    02134
7    90405
8    60614
9    90210
Name: zip, dtype: str

## Step 7: Standardize the Dates

In [6]:
df['date'] = pd.to_datetime(df['date'], format='mixed')
print('Column dtypes after date conversion:')
df.dtypes

Column dtypes after date conversion:


order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object

**Checkpoint:** Shape is (292, 6), zips are 5-character strings, dates are `datetime64`. ✓

## Step 8: Investigate the Negative Quantities

In [7]:
print('Rows with negative qty:')
display(df[df['qty'] < 0])

# Decision: keep these rows as-is — negative quantities represent product returns.
# The quantities are small whole numbers (-2, -4, -5), not the kind of
# extreme outlier a data-entry typo would produce. Real retail systems
# use negative qty to record returns/reversals against the same order record.
# Removing them would erase valid transaction history and could cause a
# business to overcount revenue or undercount return rates.
# Flipping them to positive would misrepresent the direction of the transaction.
# We document them here and leave them intact for downstream analysts to handle
# at the reporting layer (e.g., separate returns from sales in a pivot table).

print('\nDecision: KEEP as returns (negative qty preserved).')
print('These are treated as legitimate return transactions.')
print(f'Final shape: {df.shape}')

Rows with negative qty:


,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116



Decision: KEEP as returns (negative qty preserved).
These are treated as legitimate return transactions.
Final shape: (292, 6)


## Step 9: Save the Cleaned File

In [8]:
df.to_csv('sales_clean.csv', index=False)
print('sales_clean.csv saved successfully.')
print(f'Final shape: {df.shape}')
display(df.head(3))

sales_clean.csv saved successfully.
Final shape: (292, 6)


,order_id,date,product,price,qty,zip
0,1254,2026-04-06,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,2026-05-06,mug,5.50,1,10001


## Step 10: Cleaning Log

### Cleaning Log

| # | What | Why | Row / value count |
|---|------|-----|-------------------|
| 1 | Loaded `messy_sales.csv` | Establish baseline | **300 rows loaded**, 6 columns |
| 2 | Imputed 12 missing `price` values with the **median ($37.53)** | Price distributions are right-skewed — a handful of high-ticket items (monitors at $200–$400+) pull the mean up. The median resists that pull and better represents what a "typical" order costs, so it is the safer imputed value. | **12 prices filled**; 0 missing after |
| 3 | Removed **8 duplicate rows** with `.drop_duplicates()` | Exact-row duplicates are data-entry or system errors; keeping them inflates order counts and revenue figures. | **292 rows remaining** |
| 4 | Converted `zip` from `int64` to 5-character zero-padded string | Integer storage silently drops leading zeros (e.g., Boston zip `02134` became `2134`). Text storage with `.str.zfill(5)` restores them so geographic joins or lookups work correctly. | All 292 zip codes corrected |
| 5 | Converted `date` from mixed text formats to `datetime64` using `pd.to_datetime(..., format='mixed')` | Two date formats coexisted (`MM/DD/YYYY` and `YYYY-MM-DD`). Converting to a real datetime type enables sorting, filtering by date range, and time-series aggregation. | All 292 dates standardized |
| 6 | **Kept** 3 rows with negative `qty` values (-5, -4, -2) | These appear to be product returns recorded against the original order. The values are small, plausible return quantities (not data-entry typos like `-500`). Removing them would erase return history; flipping them positive would mislabel returns as sales. They are preserved for downstream reporting. | 3 return rows retained; shape stays (292, 6) |

**How a wrong choice here could mislead a business decision:** If we had used the mean instead of the median to fill the 12 missing prices, those orders would have been assigned inflated values (pulled up by high-ticket monitors and keyboards), overstating revenue for the periods those orders fall in — and a manager might conclude a product segment is more profitable than it actually is.

---
## Appendix: AI Use Disclosure

**Tool used:** Claude (Anthropic) via the Claude in Chrome side panel 
**Role:** Assisted with code structure, syntax for `.str.zfill()`, `pd.to_datetime(format='mixed')`, and notebook formatting. 
**Steps 2, 8, and 10** (observations, negative-quantity justification, and cleaning log reasoning) reflect my own analysis of the data. 

**Key prompts used:**
- "Help me build and run module3_lab.ipynb for this CS 82A assignment on cleaning messy_sales.csv"
- Provided the assignment instructions and data dictionary; AI handled pandas syntax and notebook structure